In [4]:
import json
import logging
import requests
from time import time

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[logging.StreamHandler(), logging.FileHandler("portfolio.log")]
)

def execution_timer(func):
    def wrapper(*args, **kwargs):
        start_time = time()
        result = func(*args, **kwargs)
        end_time = time()
        logging.info(f"Execution of '{func.__name__}' took {end_time - start_time:.2f} seconds.")
        return result
    return wrapper

class ReportSaver:
    def __init__(self, filename):
        self.filename = filename
        self.file = None

    def __enter__(self):
        self.file = open(self.filename, 'w')
        return self.file

    def __exit__(self, exc_type, exc_val, exc_tb):
        if self.file:
            self.file.close()
        logging.info("Report file successfully closed by the Context Manager.")
        return False

class Asset:
    def __init__(self, name, symbol, quantity):
        self.name = name
        self.symbol = symbol
        self._quantity = quantity

    @property
    def quantity(self):
        return self._quantity

    def get_value(self, current_price):
        raise NotImplementedError("Subclasses must implement get_value()!")

class CryptoAsset(Asset):
    def __init__(self, name, symbol, quantity, api_id):
        super().__init__(name, symbol, quantity)
        self.api_id = api_id

    def get_value(self, current_price):
        return self._quantity * current_price

class Portfolio:
    def __init__(self):
        self.assets = []

    def add_asset(self, asset):
        self.assets.append(asset)

    def __iter__(self):
        self._index = 0
        return self

    def __next__(self):
        if self._index < len(self.assets):
            current_asset = self.assets[self._index]
            self._index += 1
            return current_asset
        raise StopIteration

def summary_generator(portfolio, price_data):
    for asset in portfolio:
        price = price_data.get(asset.api_id, {}).get('usd', 0)
        value = asset.get_value(price)
        yield f"{asset.name} ({asset.symbol}): Qty {asset.quantity} | Price: ${price:,.2f} | Value: ${value:,.2f}"

@execution_timer
def fetch_live_prices(crypto_ids):
    logging.info("Connecting to CoinGecko API to fetch live prices...")
    url = "https://api.coingecko.com/api/v3/simple/price"
    params = {
        "ids": ",".join(crypto_ids),
        "vs_currencies": "usd"
    }
    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        return response.json()
    except requests.RequestException as error:
        logging.error(f"API Request failed: {error}")
        return {}

if __name__ == "__main__":
    bitcoin = CryptoAsset("Bitcoin", "BTC", 0.5, "bitcoin")
    ethereum = CryptoAsset("Ethereum", "ETH", 2.5, "ethereum")

    my_portfolio = Portfolio()
    my_portfolio.add_asset(bitcoin)
    my_portfolio.add_asset(ethereum)

    crypto_keys = [asset.api_id for asset in my_portfolio]
    live_prices = fetch_live_prices(crypto_keys)

    report_data = {
        "portfolio_summary": [],
        "total_value": 0.0
    }

    print("\n--- Live Portfolio Tracking ---")
    
    for line in summary_generator(my_portfolio, live_prices):
        print(line)
        report_data["portfolio_summary"].append(line)

    total = sum(
        asset.get_value(live_prices.get(asset.api_id, {}).get('usd', 0))
        for asset in my_portfolio
    )
    report_data["total_value"] = total
    print(f"Total Portfolio Value: ${total:,.2f}\n")

    with ReportSaver("portfolio_report.json") as file:
        json.dump(report_data, file, indent=4)

2026-07-14 18:37:21,084 - INFO - Connecting to CoinGecko API to fetch live prices...
2026-07-14 18:37:21,986 - INFO - Execution of 'fetch_live_prices' took 0.90 seconds.
2026-07-14 18:37:21,991 - INFO - Report file successfully closed by the Context Manager.



--- Live Portfolio Tracking ---
Bitcoin (BTC): Qty 0.5 | Price: $63,778.00 | Value: $31,889.00
Ethereum (ETH): Qty 2.5 | Price: $1,859.98 | Value: $4,649.95
Total Portfolio Value: $36,538.95

